# gmx_MMPBSA API analysis with seaborn

This local Jupyter notebook loads a `gmx_MMPBSA` result with the Python API, extracts energy data as pandas objects, and plots the results with seaborn. It defaults to the bundled API example result, but you can point `RESULT_FILE` to any `COMPACT_MMXSA_RESULTS.mmxsa` or `_GMXMMPBSA_info` file.

## Requirements

Run this notebook in an environment where `gmx_MMPBSA` and its Python dependencies are installed. The repository environment already includes pandas, matplotlib, and seaborn in the documented dependency stack.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "GMXMMPBSA").is_dir() and (candidate / "examples").is_dir():
            return candidate
    raise RuntimeError("Could not find the gmx_MMPBSA repository root from the current working directory.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from GMXMMPBSA import API

print("Repository root:", REPO_ROOT)

## Optional: run a local gmx_MMPBSA calculation

Use this section when you want the notebook to run a system from local files before loading the results with the API. Keep `RUN_LOCAL_CALC = False` when you only want to analyze an existing `.mmxsa` or `_GMXMMPBSA_info` result.

In [ ]:
# Set this to True only when you want this notebook to launch gmx_MMPBSA.
RUN_LOCAL_CALC = True

# Folder containing mmpbsa.in and all input files.
LOCAL_WORKDIR = REPO_ROOT / "examples" / "Protein_ligand" / "ST"

# Single-trajectory GROMACS workflow. Edit these names for your system.
LOCAL_INPUT_FILE = "mmpbsa.in"
LOCAL_COMPLEX_STRUCTURE = "com.tpr"
LOCAL_COMPLEX_TRAJECTORY = "com_traj.xtc"
LOCAL_INDEX_FILE = "index.ndx"
LOCAL_RECEPTOR_GROUP = 3
LOCAL_LIGAND_GROUP = 4
LOCAL_COMPLEX_TOPOLOGY = "topol.top"
LOCAL_LIGAND_MOL2 = ""
LOCAL_COMPLEX_REFERENCE = ""

LOCAL_OUTPUT_DAT = "FINAL_RESULTS_MMPBSA.dat"
LOCAL_OUTPUT_CSV = "FINAL_RESULTS_MMPBSA.csv"
LOCAL_NUM_PROCESSORS = 1


In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

LOCAL_WORKDIR = Path(LOCAL_WORKDIR)
LOCAL_RESULT_FILE = LOCAL_WORKDIR / "COMPACT_MMXSA_RESULTS.mmxsa"

def require_local_file(relative_path, label):
    path = LOCAL_WORKDIR / relative_path
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path

if RUN_LOCAL_CALC:
    require_local_file(LOCAL_INPUT_FILE, "Input file")
    require_local_file(LOCAL_COMPLEX_STRUCTURE, "Complex structure")
    require_local_file(LOCAL_COMPLEX_TRAJECTORY, "Complex trajectory")
    require_local_file(LOCAL_INDEX_FILE, "Index file")
    if LOCAL_COMPLEX_TOPOLOGY:
        require_local_file(LOCAL_COMPLEX_TOPOLOGY, "Complex topology")
    if LOCAL_LIGAND_MOL2:
        require_local_file(LOCAL_LIGAND_MOL2, "Ligand mol2 file")
    if LOCAL_COMPLEX_REFERENCE:
        require_local_file(LOCAL_COMPLEX_REFERENCE, "Complex reference structure")

    gmx_args = [
        "gmx_MMPBSA", "-O",
        "-i", LOCAL_INPUT_FILE,
        "-cs", LOCAL_COMPLEX_STRUCTURE,
        "-ct", LOCAL_COMPLEX_TRAJECTORY,
        "-ci", LOCAL_INDEX_FILE,
        "-cg", str(LOCAL_RECEPTOR_GROUP), str(LOCAL_LIGAND_GROUP),
        "-o", LOCAL_OUTPUT_DAT,
        "-eo", LOCAL_OUTPUT_CSV,
        "-nogui",
    ]
    if LOCAL_COMPLEX_TOPOLOGY:
        gmx_args.extend(["-cp", LOCAL_COMPLEX_TOPOLOGY])
    if LOCAL_LIGAND_MOL2:
        gmx_args.extend(["-lm", LOCAL_LIGAND_MOL2])
    if LOCAL_COMPLEX_REFERENCE:
        gmx_args.extend(["-cr", LOCAL_COMPLEX_REFERENCE])

    if int(LOCAL_NUM_PROCESSORS) > 1:
        cmd = ["mpirun", "-np", str(LOCAL_NUM_PROCESSORS)] + gmx_args
    else:
        cmd = gmx_args

    env = os.environ.copy()
    print("Working directory:", LOCAL_WORKDIR)
    print("Running:", " ".join(shlex.quote(part) for part in cmd))
    completed = subprocess.run(
        cmd,
        cwd=LOCAL_WORKDIR,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    if completed.returncode:
        log_path = LOCAL_WORKDIR / "gmx_MMPBSA.log"
        if log_path.exists():
            print("Last lines from gmx_MMPBSA.log:")
            print("".join(log_path.read_text(errors="replace").splitlines(keepends=True)[-120:]))
        raise RuntimeError(f"gmx_MMPBSA failed with exit code {completed.returncode}")

    if not LOCAL_RESULT_FILE.exists():
        raise FileNotFoundError(f"Calculation finished, but no compact result file was found: {LOCAL_RESULT_FILE}")
    RESULT_FILE = LOCAL_RESULT_FILE
    print("Result file for API analysis:", RESULT_FILE)
else:
    print("RUN_LOCAL_CALC is False; skipping local calculation.")
    print("Set RESULT_FILE manually in the next section, or set RUN_LOCAL_CALC = True and rerun this cell.")


## 1. Load a result file

The compact `.mmxsa` file is the most portable input for API analysis. Loading `_GMXMMPBSA_info` also works when the referenced output/topology files are still in place.

In [ ]:
RESULT_FILE = globals().get("RESULT_FILE", REPO_ROOT / "examples" / "API" / "COMPACT_MMXSA_RESULTS.mmxsa")

def load_result(result_file):
    result_file = Path(result_file)
    if hasattr(API, "load"):
        return API.load(result_file)
    api = API.MMPBSA_API()
    api.load_file(result_file)
    return api

api = load_result(RESULT_FILE)
info = api.get_info()
inputs = api.get_input()
energy = api.get_energy()

print("Loaded:", RESULT_FILE)
print("Frames:", info.get("numframes"))
print("Temperature:", inputs.get("general", {}).get("temperature"))
print("Energy map:")
print(energy["map"])

## 2. Select an energy table

For the default protein-ligand example, the main binding table is `normal / gb / delta`. Change these keys based on the printed energy map for other calculations.

In [ ]:
ETYPE = "normal"
MODEL = "gb"
MOLECULE = "delta"

energy_table = energy["data"][ETYPE][MODEL][MOLECULE].copy()
summary_table = energy["summary"][ETYPE][MODEL][MOLECULE].copy()

display(energy_table.head())
display(summary_table)

## 3. Prepare tidy plotting data

In [ ]:
SUMMARY_ROWS = {"Average", "SD", "SEM"}
frame_table = energy_table.loc[[idx for idx in energy_table.index if str(idx) not in SUMMARY_ROWS]].copy()
frame_table.index.name = "Frame"

preferred_terms = ["VDWAALS", "EEL", "EGB", "ESURF", "GGAS", "GSOLV", "TOTAL"]
terms = [term for term in preferred_terms if term in frame_table.columns]

tidy = (
    frame_table[terms]
    .reset_index()
    .melt(id_vars="Frame", var_name="Energy term", value_name="Energy")
)
tidy["Frame"] = pd.to_numeric(tidy["Frame"], errors="coerce")

display(tidy.head())

## 4. Plot per-frame energy terms

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=tidy, x="Frame", y="Energy", hue="Energy term", marker="o")
plt.title(f"{ETYPE} / {MODEL} / {MOLECULE} energy terms")
plt.ylabel("Energy")
plt.tight_layout()
plt.show()

## 5. Plot average energies with SEM

In [ ]:
summary_plot = pd.DataFrame({
    "Energy term": terms,
    "Average": [summary_table.loc["Average", term] for term in terms],
    "SEM": [summary_table.loc["SEM", term] for term in terms],
})

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=summary_plot, x="Energy term", y="Average", color="#4c78a8")
ax.errorbar(
    x=range(len(summary_plot)),
    y=summary_plot["Average"],
    yerr=summary_plot["SEM"],
    fmt="none",
    c="black",
    capsize=4,
)
plt.title(f"{ETYPE} / {MODEL} / {MOLECULE} average energies")
plt.ylabel("Average energy")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

display(summary_plot)

## 6. Export selected values

In [ ]:
OUTPUT_CSV = REPO_ROOT / "examples" / "API" / "gb_delta_total_from_notebook.csv"

if "TOTAL" in energy_table.columns:
    total = energy_table["TOTAL"].copy()
    total.to_frame("GB delta TOTAL").to_csv(OUTPUT_CSV)
    print("Wrote:", OUTPUT_CSV)
    display(total.to_frame("GB delta TOTAL").head())
else:
    print("TOTAL was not found in this energy table. Available columns:")
    print(list(energy_table.columns))